# 2020 FABLE Notebook Interface Example

This notebook shows the same notebook-facing workflow on a production-size generated Modelwright model from the public 2020 FABLE Calculator benchmark workbook.

The original workbook is not tracked in this repository. The tracked artifact is Modelwright's generated Python output, compressed as `examples/fable_2020/generated_fable_2020_model.py.xz`. The example script decompresses that generated model into ignored `tmp/` working space before import.

## 1. Make The Source Checkout Importable

This notebook is meant to be opened from a source checkout. The setup cell finds the repository root and places it on `sys.path`, so imports from `examples/` resolve even if the notebook file is copied under `tmp/notebooks/`.

Expected output: the package project name, `modelwright`.

In [1]:
from pathlib import Path
import sys
import tomllib

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

if not (repo_root / "pyproject.toml").exists():
    raise RuntimeError("Could not find the Modelwright repository root.")

project_name = tomllib.loads((repo_root / "pyproject.toml").read_text())["project"]["name"]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Using project: {project_name}")

Using project: modelwright


## 2. Load The Production-Size Example Facade

`build_facade()` imports the generated FABLE Python model and declares a small analyst-facing wrapper around three validated `SCENARIOS selection` outputs. The first import may take time because the generated model is intentionally production-sized.

Expected output: the compressed archive path, the decompressed generated-model path under `tmp/`, and the declared output names.

In [2]:
from examples.fable_2020.notebook_interface import ARCHIVE_PATH, MODEL_PATH, build_facade
from modelwright.notebooks import outputs_frame, report_frames, table_frame

facade = build_facade()

print(f"Archive: {ARCHIVE_PATH.relative_to(repo_root)}")
print(f"Generated model path: {MODEL_PATH}")
print(f"Declared outputs: {list(facade.outputs())}")

Archive: examples/fable_2020/generated_fable_2020_model.py.xz
Generated model path: tmp/examples/fable_2020/generated_fable_2020_model.py
Declared outputs: ['scenario_metric_1', 'scenario_metric_2', 'scenario_metric_3']


## 3. Calculate The Wrapped Model

This cell calls the generated model through the facade. The stored output records the three wrapped FABLE outputs. The Phase 26 full-validation evidence for the underlying generated model remains: 281,741 comparable cached outputs, 281,741 matches, and 0 mismatches.

Expected output: three `SCENARIOS selection` values.

In [3]:
values = facade.calculate()
{cell_ref: values[cell_ref] for cell_ref in [
    "SCENARIOS selection!D20",
    "SCENARIOS selection!D21",
    "SCENARIOS selection!D22",
]}

{'SCENARIOS selection!D20': 2.146115426018433,
 'SCENARIOS selection!D21': 1.8982220554032356,
 'SCENARIOS selection!D22': 1.462761288724012}

## 4. Display Wrapped Outputs As A DataFrame

The notebook helper presents the declared outputs with human names while preserving raw cell references for provenance.

Expected output: a three-row DataFrame for the wrapped scenario metrics.

In [4]:
outputs_frame(facade)[["name", "cell_ref", "value"]]

                name                 cell_ref     value
0  scenario_metric_1  SCENARIOS selection!D20  2.146115
1  scenario_metric_2  SCENARIOS selection!D21  1.898222
2  scenario_metric_3  SCENARIOS selection!D22  1.462761

## 5. Display A Declared FABLE Table Slice

The wrapper declares `SCENARIOS selection!D20:D22` as a rectangular table with row labels. This turns a raw output slice into an inspectable table.

Expected output: the three selected FABLE values as a one-column DataFrame.

In [5]:
table_frame(facade, "scenario_selection_slice")

        value
row          
d20  2.146115
d21  1.898222
d22  1.462761

## 6. Keep Provenance Available

DataFrame display is the humane interface, but workbook provenance is still available. Table frame metadata stores the sheet, range, and cell references used to build the view.

Expected output: the original workbook sheet and range.

In [6]:
scenario_table = table_frame(facade, "scenario_selection_slice")
{
    "sheet": scenario_table.attrs["sheet"],
    "range_ref": scenario_table.attrs["range_ref"],
    "cell_refs": scenario_table.attrs["cell_refs"],
}

{'sheet': 'SCENARIOS selection',
 'range_ref': 'D20:D22',
 'cell_refs': [['SCENARIOS selection!D20'],
  ['SCENARIOS selection!D21'],
  ['SCENARIOS selection!D22']]}

## 7. Bundle Outputs Into A Report

Reports collect declared cells and tables under one human-facing name. This gives a notebook user a repeatable reporting boundary without editing the generated model.

Expected output: the report's cell DataFrame.

In [7]:
frames = report_frames(facade, "scenario_selection")
frames["cells"][["name", "cell_ref", "value"]]

                name                 cell_ref     value
0  scenario_metric_1  SCENARIOS selection!D20  2.146115
1  scenario_metric_2  SCENARIOS selection!D21  1.898222
2  scenario_metric_3  SCENARIOS selection!D22  1.462761